# Comparacion de errores ARIMA y N-BEATS


Este notebook compara las metricas exportadas por `06_arima_error_metrics.ipynb` y `07_nbeats_error_metrics.ipynb`. No recalcula forecasts: solo consolida MAE, RMSE y MAPE, valida que ambos modelos evaluen los mismos tramos temporales por horizonte y exporta tablas listas para analisis o reporte.


Importamos las dependencias necesarias para leer las metricas, construir tablas comparativas, exportar resultados y graficar barras por horizonte.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)


Definimos las rutas de entrada y salida. Este notebook espera que ya existan las metricas ARIMA y N-BEATS exportadas por los notebooks anteriores.


In [ ]:
DATASET_NAME = "AT"
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
OUTPUT_DIR = ARTIFACTS_DIR / "comparison"

ARIMA_METRICS_PATH = OUTPUT_DIR / f"{DATASET_NAME.lower()}_arima_error_metrics.csv"
NBEATS_METRICS_PATH = OUTPUT_DIR / f"{DATASET_NAME.lower()}_nbeats_error_metrics.csv"

ARIMA_METRICS_PATH, NBEATS_METRICS_PATH


Cargamos las metricas de ambos modelos y verificamos que los archivos existan antes de construir la comparacion.


In [ ]:
missing_inputs = [path for path in [ARIMA_METRICS_PATH, NBEATS_METRICS_PATH] if not path.exists()]
if missing_inputs:
    missing = "\n".join(str(path.relative_to(NOTEBOOK_DIR)) for path in missing_inputs)
    raise FileNotFoundError(
        "Faltan metricas para comparar. Ejecuta primero 06_arima_error_metrics.ipynb y 07_nbeats_error_metrics.ipynb.\n"
        f"Archivos faltantes:\n{missing}"
    )

metrics_df = pd.concat(
    [pd.read_csv(ARIMA_METRICS_PATH), pd.read_csv(NBEATS_METRICS_PATH)],
    ignore_index=True,
)
metrics_df = metrics_df.sort_values(["horizon", "model"]).reset_index(drop=True)
metrics_df


Validamos que cada experimento tenga ambos modelos y que compartan el mismo tramo temporal de prueba antes de comparar sus errores.


In [ ]:
required_models = {"ARIMA", "N-BEATS"}
validation_rows = []

for experiment, group in metrics_df.groupby("experiment"):
    models = set(group["model"])
    missing_models = sorted(required_models.difference(models))
    if missing_models:
        raise ValueError(f"Faltan modelos para {experiment}: {missing_models}")

    test_starts = group["test_start"].nunique()
    test_ends = group["test_end"].nunique()
    test_rows = group["test_rows"].nunique()
    if test_starts != 1 or test_ends != 1 or test_rows != 1:
        raise ValueError(f"ARIMA y N-BEATS no estan alineados temporalmente para {experiment}.")

    validation_rows.append({
        "experiment": experiment,
        "horizon": int(group["horizon"].iloc[0]),
        "test_rows": int(group["test_rows"].iloc[0]),
        "test_start": group["test_start"].iloc[0],
        "test_end": group["test_end"].iloc[0],
    })

validation_df = pd.DataFrame(validation_rows).sort_values("horizon").reset_index(drop=True)
validation_df


Construimos una tabla lado a lado con MAE, RMSE y MAPE para ARIMA y N-BEATS por horizonte.


In [ ]:
comparison_df = (
    metrics_df.pivot_table(
        index=["dataset", "experiment", "label", "horizon", "test_rows", "test_start", "test_end"],
        columns="model",
        values=["mae", "rmse", "mape"],
        aggfunc="first",
    )
    .sort_index(level="horizon")
)
comparison_df


Calculamos diferencias relativas para identificar en que horizontes N-BEATS mejora o empeora respecto a ARIMA. Valores negativos indican menor error de N-BEATS.


In [ ]:
wide_df = comparison_df.reset_index()
wide_df.columns = ["_".join(column).strip("_") if isinstance(column, tuple) else column for column in wide_df.columns]

for metric in ["mae", "rmse", "mape"]:
    arima_column = f"{metric}_ARIMA"
    nbeats_column = f"{metric}_N-BEATS"
    wide_df[f"{metric}_nbeats_minus_arima"] = wide_df[nbeats_column] - wide_df[arima_column]
    wide_df[f"{metric}_relative_change_pct"] = (wide_df[nbeats_column] / wide_df[arima_column] - 1) * 100

wide_df


Exportamos las metricas consolidadas, la tabla comparativa y la version plana con diferencias relativas para uso posterior.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = OUTPUT_DIR / f"{DATASET_NAME.lower()}_model_error_metrics.csv"
comparison_path = OUTPUT_DIR / f"{DATASET_NAME.lower()}_model_error_comparison.csv"
wide_path = OUTPUT_DIR / f"{DATASET_NAME.lower()}_model_error_comparison_wide.csv"
validation_path = OUTPUT_DIR / f"{DATASET_NAME.lower()}_model_error_alignment.csv"

metrics_df.to_csv(metrics_path, index=False)
comparison_df.to_csv(comparison_path)
wide_df.to_csv(wide_path, index=False)
validation_df.to_csv(validation_path, index=False)

print(f"metrics: {metrics_path.relative_to(NOTEBOOK_DIR)}")
print(f"comparison: {comparison_path.relative_to(NOTEBOOK_DIR)}")
print(f"wide comparison: {wide_path.relative_to(NOTEBOOK_DIR)}")
print(f"alignment: {validation_path.relative_to(NOTEBOOK_DIR)}")


Graficamos MAE, RMSE y MAPE por horizonte para comparar visualmente ambos modelos.


In [ ]:
for metric in ["mae", "rmse", "mape"]:
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_df = metrics_df.pivot(index="label", columns="model", values=metric).loc[wide_df["label"]]
    plot_df.plot(kind="bar", ax=ax)
    ax.set_title(metric.upper())
    ax.set_xlabel("horizonte")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=0)
    plt.show()
